### Câu 1

In [10]:
import sqlite3

# Tạo kết nối tới database trong RAM
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng
cursor.execute('''
    CREATE TABLE data (
        A REAL,
        B REAL
    )
''')

# Thêm dữ liệu mẫu
data = [
    (10, 12),
    (20, 24),
    (30, 33),
    (40, 47),
    (50, 55)
]

cursor.executemany('INSERT INTO data (A, B) VALUES (?, ?)', data)
conn.commit()


In [11]:
import math

conn.create_function("SQRT", 1, math.sqrt)

query = '''
SELECT
    (COUNT(*) * SUM(A * B) - SUM(A) * SUM(B)) /
    (SQRT(COUNT(*) * SUM(A * A) - SUM(A) * SUM(A)) *
     SQRT(COUNT(*) * SUM(B * B) - SUM(B) * SUM(B))) AS pearson_r
FROM data
'''

cursor.execute(query)
result = cursor.fetchone()
print(f"Hệ số tương quan Pearson r_AB = {result[0]:.4f}")


Hệ số tương quan Pearson r_AB = 0.9972


### Câu 2 

In [12]:
import sqlite3
import pandas as pd
from scipy.stats import chi2_contingency

# Kết nối SQLite in-memory
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng điểm (dạng quan hệ)
cursor.execute("""
CREATE TABLE scores (
    day TEXT,
    model TEXT,
    score REAL
)
""")

# Dữ liệu theo bảng trong ảnh
data = [
    ('Day 1', 'A', 8), ('Day 1', 'B', 9), ('Day 1', 'C', 7),
    ('Day 2', 'A', 7.5), ('Day 2', 'B', 8.5), ('Day 2', 'C', 7),
    ('Day 3', 'A', 6), ('Day 3', 'B', 7), ('Day 3', 'C', 8),
    ('Day 4', 'A', 7), ('Day 4', 'B', 6), ('Day 4', 'C', 5)
]

cursor.executemany("INSERT INTO scores VALUES (?, ?, ?)", data)
conn.commit()


In [13]:
# Tạo view phân nhóm điểm
cursor.execute("""
CREATE VIEW score_grouped AS
SELECT *,
    CASE
        WHEN score < 6.5 THEN 'Low'
        WHEN score <= 8 THEN 'Medium'
        ELSE 'High'
    END AS score_group
FROM scores
""")


In [14]:
# Lấy bảng tần số giữa ngày và nhóm điểm
query = """
SELECT day, score_group, COUNT(*) as count
FROM score_grouped
GROUP BY day, score_group
"""

df = pd.read_sql_query(query, conn)

# Pivot thành bảng phù hợp cho chi2
contingency = df.pivot(index='day', columns='score_group', values='count').fillna(0)

print(" Bảng tần số:")
print(contingency)


 Bảng tần số:
score_group  High  Low  Medium
day                           
Day 1         1.0  0.0     2.0
Day 2         1.0  0.0     2.0
Day 3         0.0  1.0     2.0
Day 4         0.0  2.0     1.0


In [15]:
# Kiểm định χ²
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"\n Kết quả kiểm định Chi-squared:")
print(f"Chi2 statistic = {chi2:.4f}")
print(f"p-value = {p:.4f}")
print(f"Số bậc tự do = {dof}")
print("\nBảng kỳ vọng:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns))

# Đánh giá
if p < 0.05:
    print("\n Kết luận: Có sự khác biệt đáng kể giữa các ngày.")
else:
    print("\n Kết luận: Không có sự khác biệt đáng kể giữa các ngày.")



 Kết quả kiểm định Chi-squared:
Chi2 statistic = 6.0952
p-value = 0.4126
Số bậc tự do = 6

Bảng kỳ vọng:
score_group  High   Low  Medium
day                            
Day 1         0.5  0.75    1.75
Day 2         0.5  0.75    1.75
Day 3         0.5  0.75    1.75
Day 4         0.5  0.75    1.75

 Kết luận: Không có sự khác biệt đáng kể giữa các ngày.


### Câu 3

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("CREATE TABLE flights (flight_id INTEGER, departure_time INTEGER)")
cursor.executemany("INSERT INTO flights VALUES (?, ?)", [
    (1, 830), (2, 1445), (3, 1000), (4, 50), (5, 5)
])
query3 = """
SELECT
  flight_id,
  departure_time,
  printf('%02d:%02d', departure_time / 100, departure_time % 100) AS formatted_time
FROM flights;
"""
df3 = pd.read_sql_query(query3, conn)
df3



,flight_id,departure_time,formatted_time
0,1,830,08:30
1,2,1445,14:45
2,3,1000,10:00
3,4,50,00:50
4,5,5,00:05


### Câu 4

In [18]:
cursor.execute("CREATE TABLE values_data (value REAL)")
cursor.executemany("INSERT INTO values_data (value) VALUES (?)", [
    (10,), (12,), (13,), (12.5,), (11,), (500,)  # 500 là ngoại lệ
])
# Lấy dữ liệu ra để xử lý
df_values = pd.read_sql_query("SELECT value FROM values_data", conn)

# Tính median
median = df_values['value'].median()

# MAD = median của |value - median|
mad = (df_values['value'] - median).abs().median()

# Đánh dấu ngoại lệ nếu |x - median| > 1.5 * MAD
df_values['is_outlier'] = ((df_values['value'] - median).abs() > 1.5 * mad).astype(int)
df_values


,value,is_outlier
0,10.0,1
1,12.0,0
2,13.0,0
3,12.5,0
4,11.0,0
5,500.0,1


### Câu 5

In [ ]:
cursor.execute("CREATE TABLE Patient (last_name TEXT, weight REAL, height REAL)")
cursor.executemany("INSERT INTO Patient VALUES (?, ?, ?)", [
    ("Nguyen", 60, 170),
    ("Nguyen", 60.2, 170.1),
    ("Tran", 65, 172),
    ("Le", 70, 180)
])

query5 = """
SELECT
  p1.last_name AS name1,
  p2.last_name AS name2,
  p1.weight, p2.weight,
  p1.height, p2.height,
  CASE
    WHEN p1.last_name = p2.last_name
         AND ABS(p1.weight - p2.weight) < 1
         AND ABS(p1.height - p2.height) < 1
    THEN 'Có thể cùng người'
    ELSE 'Khác người'
  END AS same_person
FROM Patient p1
JOIN Patient p2
  ON p1.rowid < p2.rowid;
"""
df5 = pd.read_sql_query(query5, conn)
df5


,name1,name2,weight,weight,height,height,same_person
0,Nguyen,Nguyen,60.0,60.2,170.0,170.1,Có thể cùng người
1,Nguyen,Tran,60.0,65.0,170.0,172.0,Khác người
2,Nguyen,Le,60.0,70.0,170.0,180.0,Khác người
3,Nguyen,Tran,60.2,65.0,170.1,172.0,Khác người
4,Nguyen,Le,60.2,70.0,170.1,180.0,Khác người
5,Tran,Le,65.0,70.0,172.0,180.0,Khác người
